# Agente revisor de rendiciones de gastos — Tarea Final Curso 1 (Grupo 05)

Este notebook **solo importa** desde `src/`: la lógica vive en los módulos.
Ejecútalo de arriba abajo desde la carpeta `tarea-final-c1/`. Todos los datos son ficticios.

## 0. Ficha de reproducción

Modelo, temperatura, versiones de librerías, variables de entorno y datos usados.
La tabla completa está en `README.md`. Esta celda muestra lo que realmente está instalado.

In [ ]:
from importlib.metadata import version

from src.config import MODELO_DEL_CURSO, TEMPERATURA

print('Modelo:', MODELO_DEL_CURSO, '| temperatura:', TEMPERATURA)
for libreria in ['langgraph', 'langchain', 'langchain-google-genai', 'langchain-redis', 'redis', 'redisvl']:
    print(f'{libreria}=={version(libreria)}')

## 1. Caso y criterio de éxito

**Problema.** En **Comercial Los Alerces SpA** (empresa ficticia), cada colaborador elige la política y la categoría de sus gastos. Es común que elija mal, porque una misma categoría (peaje, combustible, almuerzo) existe en varias políticas, y cada una tiene un tratamiento contable distinto. Clasificar mal genera cuentas equivocadas, reportes distorsionados y reprocesos.

**Usuario.** El analista de Control de Gestión que hace la primera revisión normativa. El colaborador que rinde recibe la observación y corrige.

**Entrada.** Un gasto rendido: fechas, monto, tipo de documento, glosa, política y categoría declaradas, unidad de negocio, fondo y campos obligatorios. También acepta preguntas sobre la política. Lo declarado es una pista, no la respuesta.

**Alcance.** Revisa gastos de a uno contra la política POL-RG-01 v1.6. Puede buscar en la política (RAG), validar reglas exactas, emitir un veredicto y registrarlo en una base de prueba. No paga, no borra registros, no aprueba sin validar y no responde temas ajenos a las rendiciones.

**Salida esperada.** Una decisión (aprobar, observar o rechazar), con la política, categoría y cuenta correctas, los códigos de las reglas y el motivo. Si corresponde, el saldo del fondo actualizado y el registro del veredicto.

**Comprobación.** Un peaje con auto propio declarado en Viajes Internos (P02) debe quedar observado y reclasificado a P01 / Peajes / CTA-5100, citando RG-09. El criterio general es el golden set de 34 casos: el evaluador los corre con Gemini real, y la corrida final debe pasar todos.

**Por qué un agente.** El LLM interpreta la glosa y la contrasta con la política. Las reglas exactas se calculan con código, para que sean auditables. El ciclo ReAct permite buscar, validar, corregir y decidir, o detenerse de forma segura.

Detalle completo en [docs/caso_y_criterio.md](docs/caso_y_criterio.md).

## 2. Entorno y carga de configuración

In [ ]:
import os

from src.config import CARPETA_PROYECTO, obtener_llm, registrar_traza
from src.contrato import MAX_PASOS, DECISIONES

# Mostrar solo el nombre de la carpeta: no imprimir la ruta con el usuario del sistema.
print('Proyecto:', CARPETA_PROYECTO.name)
print('GEMINI_MODEL:', os.getenv('GEMINI_MODEL') or 'FALTA')
print('EMBEDDING_MODEL:', os.getenv('EMBEDDING_MODEL') or 'FALTA')
print('GOOGLE_API_KEY:', 'configurada (valor oculto)' if os.getenv('GOOGLE_API_KEY', '').strip() else 'FALTA')
print('REDIS_URL:', 'configurada (valor oculto)' if os.getenv('REDIS_URL', '').strip() else 'FALTA')
print('REDIS_PREFIJO:', os.getenv('REDIS_PREFIJO') or 'FALTA')
print('MOSTRAR_TRAZAS:', os.getenv('MOSTRAR_TRAZAS', '1'))
print('MAX_PASOS:', MAX_PASOS, '| decisiones:', ', '.join(DECISIONES))

## 3. Caso normal: ReAct con herramienta, observación, decisión y parada

La traza debe mostrar la búsqueda, su observación, la validación, su observación y la respuesta final.

In [ ]:
import contextlib
import csv
import io
import json
import os
import tempfile
import re
from pathlib import Path

from src import registro
from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar_con_caso

# G-001 recorre ReAct. Capturamos registrar_traza y la mostramos en la salida de Jupyter.
with (CARPETA_PROYECTO / 'datos' / 'rendiciones_prueba.csv').open(encoding='utf-8', newline='') as archivo:
    gastos = list(csv.DictReader(archivo))
gasto_normal = next(gasto for gasto in gastos if gasto['id'] == 'G-001')
gasto_normal['monto'] = int(gasto_normal['monto'])
gasto_normal['vb_gerente_general'] = gasto_normal['vb_gerente_general'].strip().casefold() == 'true'
for campo in ('patente', 'numero_ot'):
    gasto_normal[campo] = gasto_normal[campo] or None

ruta_registro_original = registro.RUTA_REGISTRO
trazas_originales = os.environ.get('MOSTRAR_TRAZAS')
try:
    with tempfile.TemporaryDirectory() as carpeta_temporal:
        registro.RUTA_REGISTRO = Path(carpeta_temporal) / 'registro_g001.db'
        os.environ['MOSTRAR_TRAZAS'] = '1'
        agente_normal = construir_agente()
        hilo_normal = 'DEMO-REACT-G001'
        mensaje_normal = 'Revisa este gasto rendido: ' + json.dumps(gasto_normal, ensure_ascii=False)
        salida_traza = io.StringIO()
        with contextlib.redirect_stdout(salida_traza):
            respuesta_normal = conversar_con_caso(agente_normal, mensaje_normal, hilo_normal, gasto_normal)
        traza_normal = salida_traza.getvalue()
        estado_normal = agente_normal.get_state({'configurable': {'thread_id': hilo_normal}}).values
finally:
    registro.RUTA_REGISTRO = ruta_registro_original
    if trazas_originales is None:
        os.environ.pop('MOSTRAR_TRAZAS', None)
    else:
        os.environ['MOSTRAR_TRAZAS'] = trazas_originales

def abreviar_traza(valor, limite=180):
    """Acorta textos extensos (por ejemplo, fragmentos) sin alterar la ejecución guardada."""
    if isinstance(valor, dict):
        return {clave: abreviar_traza(contenido, limite) for clave, contenido in valor.items()}
    if isinstance(valor, list):
        return [abreviar_traza(item, limite) for item in valor]
    if isinstance(valor, str) and len(valor) > limite:
        return valor[:limite] + '… [texto resumido]'
    return valor

def resumir_eventos(traza):
    """Muestra una línea por evento real, con los textos largos acotados."""
    patron = r"(?ms)^── \[(?P<evento>[^\]]+)\] ──\n(?P<datos>.*?)(?=^── \[|\Z)"
    eventos = list(re.finditer(patron, traza))
    for coincidencia in eventos:
        evento = coincidencia.group('evento')
        try:
            datos = json.loads(coincidencia.group('datos').strip())
        except json.JSONDecodeError:
            datos = abreviar_traza(coincidencia.group('datos').strip())
        if evento == 'llm' and isinstance(datos, dict):
            llamadas = datos.get('pide_herramientas') or []
            datos = {'paso': datos.get('paso'), 'herramientas': [llamada.get('name') for llamada in llamadas],
                     'texto': abreviar_traza(datos.get('texto', ''), 120)}
        elif evento == 'observacion' and isinstance(datos, dict):
            datos = {'herramienta': datos.get('herramienta'),
                     'resultado': abreviar_traza(datos.get('resultado', ''), 160)}
        else:
            datos = abreviar_traza(datos)
        print(f"[{evento}] {json.dumps(datos, ensure_ascii=False, separators=(',', ':'))}")
    return eventos

print('Traza resumida de registrar_traza:')
eventos_resumidos = resumir_eventos(traza_normal)
print('Respuesta final:', respuesta_normal)
assert any(e.group('evento') == 'llm' for e in eventos_resumidos), 'La traza del LLM debe verse en el notebook.'
assert any(e.group('evento') == 'observacion' and 'validar_gasto' in e.group('datos') for e in eventos_resumidos), 'La traza debe mostrar la observación de validar_gasto.'
for campo in ('gasto', 'clasificacion', 'validacion', 'veredicto'):
    print(f'{campo}: {estado_normal.get(campo)}')

## 4. Segundo turno: uso del historial

In [ ]:
from src.grafo import construir_agente, conversar

# Un agente conserva el historial por thread_id; cada caso usa un hilo distinto.
agente_historial = construir_agente()

# GS-31: un saludo breve debe responderse sin herramientas.
respuesta_gs31 = conversar(agente_historial, 'Hola', 'GS-31')
print('GS-31:', respuesta_gs31)

# GS-32: los dos turnos usan exactamente el mismo hilo.
hilo_gs32 = 'GS-32'
respuesta_turno_1 = conversar(agente_historial, 'Soy Pedro, chofer del Centro de Distribución', hilo_gs32)
respuesta_turno_2 = conversar(agente_historial, '¿Cómo me llamo y en qué unidad trabajo?', hilo_gs32)
print('GS-32 turno 1:', respuesta_turno_1)
print('GS-32 turno 2:', respuesta_turno_2)

assert 'pedro' in respuesta_turno_2.casefold(), 'El segundo turno debe recordar el nombre.'
assert any(texto in respuesta_turno_2.casefold() for texto in ('centro de distribución', 'centro de distribucion', 'cd')), 'El segundo turno debe recordar la unidad.'
# En las trazas de ambos turnos debe verse pide_herramientas vacío.

## 5. Seguridad básica: fuera de alcance y jailbreak

In [ ]:
import json
from pathlib import Path

from src.config import CARPETA_PROYECTO, MODELO_DEL_CURSO
from src.seguridad import PROMPT_SISTEMA

print('PROMPT_SISTEMA vigente:')
print(PROMPT_SISTEMA)


# Evidencia histórica real de corrida 3 y prueba dirigida PR #41; no llama servicios.
corrida = json.loads((CARPETA_PROYECTO / 'pruebas' / 'resultados' / 'corrida_3.json').read_text(encoding='utf-8'))
dirigido = json.loads((CARPETA_PROYECTO / 'pruebas' / 'resultados' / 'prueba_dirigida_PR41_general.json').read_text(encoding='utf-8'))
casos_corrida = {caso['id']: caso for caso in corrida['casos']}
print('Evidencia histórica de corrida 3:', corrida['resumen']['pasan'], '/', corrida['resumen']['total'],
      '| índice:', corrida['indice'], '| commit medido:', corrida['commit'])

for codigo in ('GS-27', 'GS-28', 'GS-29', 'GS-30'):
    caso = casos_corrida[codigo]
    turno = caso['turnos'][0]
    obtenido = turno['obtenido']
    print(f"\n{codigo}: {caso['descripcion']} | pasa={caso['pasa']}")
    print('Respuesta histórica:', obtenido['texto'])
    print('Decisión/veredicto:', obtenido['decision'], '| política:', obtenido['politica'])
    print('Herramientas:', obtenido['herramientas'])
    prohibidas = turno['esperado'].get('acciones_prohibidas', [])
    if prohibidas:
        print('Acciones indebidas comprobadas por el evaluador:', prohibidas, '→ caso pasó')
    assert caso['pasa'], f"{codigo} falló en la evidencia archivada"

jailbreak = dirigido['jailbreak']
print('\nJailbreak histórico sin gasto:', jailbreak['mensaje'])
print('Respuesta:', jailbreak['respuesta'])
print('Declara decisión:', jailbreak['declara_decision'], '| veredicto:', jailbreak['veredicto_en_estado'])
print('Herramientas:', jailbreak['herramientas'], '| filas en registro temporal:', len(jailbreak['filas_en_registro_temporal']))
assert not jailbreak['declara_decision']
assert jailbreak['veredicto_en_estado'] is None
assert not jailbreak['herramientas'] and jailbreak['filas_en_registro_temporal'] == []

# Ejecución nueva opcional: activar solo con cuota coordinada con Alex.
EJECUTAR_SEGURIDAD_EN_VIVO = False
if EJECUTAR_SEGURIDAD_EN_VIVO:
    from pruebas.evaluar import evaluar, mostrar_tabla
    resultados_seguridad = evaluar(casos=['GS-27', 'GS-28', 'GS-29', 'GS-30'], pausa=0)
    mostrar_tabla(resultados_seguridad)
else:
    print('\nNueva ejecución Gemini: omitida; se muestra la evidencia histórica verificada.')


## 6. Ampliación RAG (Redis)

In [ ]:
import json
import os
from pathlib import Path

from src.config import CARPETA_PROYECTO

# Configuración documentada para grupo05_v15; no se consulta Redis en esta celda.
print('Índice objetivo:', 'grupo05_v15')
print('Índice configurado en este entorno:', os.getenv('REDIS_PREFIJO') or 'FALTA')
print('Modelo de embeddings:', os.getenv('EMBEDDING_MODEL') or 'FALTA')
print('Esquema Redis: 3072 dimensiones, COSINE, FLAT, legacy_key_format=False')
print('Campos publicados: text, embedding, _index_name, _metadata_json; metadata: codigo, politica, fuente, texto_completo, contenido_sha256')

# Evidencia histórica auténtica, corrida 3; no llama Redis ni Gemini.
corrida_rag = json.loads((CARPETA_PROYECTO / 'pruebas' / 'resultados' / 'corrida_3.json').read_text(encoding='utf-8'))
assert corrida_rag['indice'] == 'grupo05_v15'
assert corrida_rag['resumen']['pasan'] == 34
casos_rag = {caso['id']: caso for caso in corrida_rag['casos']}

for codigo in ('GS-31', 'GS-33'):
    caso = casos_rag[codigo]
    obtenido = caso['turnos'][0]['obtenido']
    print(f"\n{codigo}: {caso['descripcion']} | respuesta histórica: {obtenido['texto']}")
    print('Herramientas:', obtenido['herramientas'])
    for fragmento in obtenido['fragmentos']:
        print('Fragmento usado:', fragmento['politica'], '| fuente:', fragmento['fuente'], '| puntaje:', fragmento['puntaje'])
    assert caso['pasa'], f"{codigo} falló en corrida 3"
    if codigo == 'GS-31':
        assert 'buscar_politica' not in obtenido['herramientas']
        assert not obtenido['fragmentos']
    else:
        assert 'buscar_politica' in obtenido['herramientas']
        assert any('RG-03' in f['fuente'] for f in obtenido['fragmentos'])
        assert '10 de noviembre' in obtenido['texto'].lower()

print('\nEtiqueta: resultados históricos reales de corrida 3; esta celda solo lee archivos y no vuelve a consultar el índice.')

# Demostración nueva opcional, solo si Alex coordina cuota/servicio y el índice está cargado.
EJECUTAR_RAG_EN_VIVO = False
if EJECUTAR_RAG_EN_VIVO:
    from pruebas.evaluar import evaluar, mostrar_tabla
    resultados_rag = evaluar(casos=['GS-31', 'GS-33'], pausa=0)
    mostrar_tabla(resultados_rag)
else:
    print('RAG en vivo: omitido para conservar cuota; ya se muestran las trazas archivadas.')


## 7. Ampliación memoria del fondo (GS-26)

In [ ]:
import json
import tempfile
from pathlib import Path

from src import registro
from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar_con_caso
from src.memoria import fondo_inicial, resumen_fondo

# GS-26 usa dos gastos del mismo fondo y el mismo hilo. Son solo dos turnos Gemini.
golden_memoria = json.loads((CARPETA_PROYECTO / 'pruebas' / 'golden_set.json').read_text(encoding='utf-8'))
caso_gs26 = next(caso for caso in golden_memoria['casos'] if caso['id'] == 'GS-26')
contexto_fondo = caso_gs26['contexto']['fondo']
fondo = fondo_inicial(contexto_fondo['id_fondo'])
assert fondo['saldo_inicial'] == contexto_fondo['saldo_inicial']
agente = construir_agente()
hilo_gs26 = 'notebook-gs26-memoria'
decisiones_gs26 = []

# El demo no modifica el registro local del usuario.
ruta_registro_original = registro.RUTA_REGISTRO
with tempfile.TemporaryDirectory() as carpeta_temporal:
    try:
        registro.RUTA_REGISTRO = Path(carpeta_temporal) / 'registro_gs26.db'
        for numero_turno, turno in enumerate(caso_gs26['turnos'], start=1):
            gasto = turno['gasto']
            mensaje = turno['mensaje'] + '\nGasto rendido:\n' + json.dumps(gasto, ensure_ascii=False)
            respuesta = conversar_con_caso(agente, mensaje, hilo_gs26, gasto, fondo)
            estado = agente.get_state({'configurable': {'thread_id': hilo_gs26}}).values
            veredicto = estado.get('veredicto')
            assert veredicto, f"Turno {numero_turno}: faltó un veredicto estructurado."
            fondo = estado['fondo']
            decisiones_gs26.append(veredicto['decision'])
            print(f"Turno {numero_turno}: {gasto['id']} -> {veredicto['decision'].upper()} | saldo {fondo['saldo_actual']:,} CLP")
            print(respuesta)
    finally:
        registro.RUTA_REGISTRO = ruta_registro_original

print(resumen_fondo(fondo))
assert decisiones_gs26 == ['aprobar', 'observar']
assert fondo['saldo_actual'] == 20000
assert fondo['gastos_aprobados'] == ['G-2601']

## 8. Ampliación acción: registro de veredictos

In [ ]:
import json
import tempfile
from pathlib import Path

from src import registro
from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar_con_caso
from src.memoria import fondo_inicial

# La escritura ocurre al cerrar una corrida real del agente; usamos SQLite temporal.
golden_registro = json.loads((CARPETA_PROYECTO / 'pruebas' / 'golden_set.json').read_text(encoding='utf-8'))
caso_registro = next(caso for caso in golden_registro['casos'] if caso['id'] == 'GS-07')
turno_registro = caso_registro['turnos'][0]
gasto_registro = turno_registro['gasto']
mensaje_registro = turno_registro['mensaje'] + '\nGasto rendido:\n' + json.dumps(gasto_registro, ensure_ascii=False)
ruta_registro_original = registro.RUTA_REGISTRO
with tempfile.TemporaryDirectory() as carpeta_temporal:
    try:
        registro.RUTA_REGISTRO = Path(carpeta_temporal) / 'registro_agente.db'
        antes_registro = registro.leer_registro()
        agente_registro = construir_agente()
        respuesta_registro = conversar_con_caso(
            agente_registro, mensaje_registro, 'notebook-registro-gs07', gasto_registro,
            fondo_inicial(gasto_registro['id_fondo']),
        )
        estado_registro = agente_registro.get_state(
            {'configurable': {'thread_id': 'notebook-registro-gs07'}}
        ).values
        despues_registro = registro.leer_registro()
        print('Registro antes:', antes_registro)
        print('Respuesta del agente:', respuesta_registro)
        print('Veredicto estructurado:', estado_registro.get('veredicto'))
        print('Registro después:', despues_registro)
        assert estado_registro.get('veredicto'), 'La corrida no produjo un veredicto estructurado.'
        assert len(despues_registro) == 1
        assert despues_registro[0]['id_gasto'] == gasto_registro['id']
        assert despues_registro[0]['decision'] == estado_registro['veredicto']['decision']
        print('El nodo de cierre registró el resultado en la base temporal; no se llamó registrar_veredicto desde el notebook.')
        registro.reiniciar_registro()
        despues_reinicio = registro.leer_registro()
        print('Registro después del reinicio temporal:', despues_reinicio)
        assert despues_reinicio == [], 'El reinicio debe vaciar solo la base temporal.'
        registro.reiniciar_registro()  # repetir el reinicio debe seguir siendo seguro
        assert registro.leer_registro() == []
        print('Segundo reinicio idempotente: la base temporal sigue vacía.')
    finally:
        registro.RUTA_REGISTRO = ruta_registro_original

## 9. Juez LLM: contrato offline y evidencia real histórica

El ejemplo offline prueba el paquete y el formato de respuesta, no la calidad del juez. GS-01 tuvo un falso negativo real (0/4 con prompts sucesivos); GS-34 rechazó correctamente una aprobación indebida en 2/2 pruebas. La evaluación real de GS-01 sigue pendiente de corrección.

In [ ]:
import json

from src.config import CARPETA_PROYECTO, MODELO_DEL_CURSO
from src.seguridad import PROMPT_JUEZ

print('Modelo del juez:', MODELO_DEL_CURSO)
print('PROMPT_JUEZ vigente:\n', PROMPT_JUEZ)

# Dictámenes reales archivados de PR #41 y decisiones finales de corrida 3.
# Se mantienen las fuentes separadas y se identifican como ejecuciones históricas.
prueba_juez = json.loads((CARPETA_PROYECTO / 'pruebas' / 'resultados' / 'prueba_dirigida_PR41_general.json').read_text(encoding='utf-8'))
corrida_juez = json.loads((CARPETA_PROYECTO / 'pruebas' / 'resultados' / 'corrida_3.json').read_text(encoding='utf-8'))
casos_finales = {caso['id']: caso for caso in corrida_juez['casos']}

for codigo, id_gasto, decision_esperada in (
    ('GS-01', 'G-0101', 'aprobar'),
    ('GS-29', 'G-2901', 'rechazar'),
):
    dictamenes = [d for d in prueba_juez['dictamenes_juez'] if d['gasto'] == id_gasto]
    assert dictamenes, f'Falta dictamen histórico de {id_gasto}'
    dictamen = dictamenes[-1]
    final = casos_finales[codigo]['turnos'][0]['obtenido']['decision']
    print(f"\n{codigo} ({id_gasto}) — dictamen real archivado PR #41:")
    print('evaluado:', dictamen['evaluado'], '| valido:', dictamen['valido'])
    print('Razón auténtica:', dictamen['razon'])
    print('Decisión aplicada por el agente en corrida 3:', final)
    assert dictamen['evaluado'] is True and dictamen['valido'] is True
    assert final == decision_esperada

print('\nEtiqueta: dictámenes históricos reales de PR #41 y decisiones finales de corrida 3; esta celda no vuelve a llamar al juez.')


## 10. Guardrail: GS-16/GS-27 simulados y control benigno Gemini

In [ ]:
from pruebas.probar_herramientas import prueba_piso_en_agente, prueba_piso_con_gemini

# La prueba completa usa el grafo real, un LLM guionado y juez que valida todo: peor caso, sin cuota.
prueba_piso_en_agente()  # GS-16, GS-01 y GS-27; SQLite temporal, sin Gemini.
print('GS-27 se simula porque no es fiable pedirle a Gemini que obedezca una glosa maliciosa; el juez se fuerza al peor caso.')

# Control benigno real: prueba de Seba con Gemini y juez reales (consume unas pocas llamadas).
# Ejecutar solo con Redis v1.6 cargado y cuota coordinada con Alex.
prueba_piso_con_gemini()

## 11. Golden set: ejecución y resultados por caso

In [ ]:
from pruebas.evaluar import evaluar, mostrar_tabla

# Muestra acotada: no ejecutes el golden completo desde el notebook.
CASOS_MUESTRA = ['GS-07']
resultados_muestra = evaluar(casos=CASOS_MUESTRA, pausa=0)
mostrar_tabla(resultados_muestra)

## 12. Verificación final

Revisión del contrato (no necesita clave de API).

In [ ]:
import subprocess
import sys

# Usa el mismo Python del notebook (su entorno virtual).
resultado = subprocess.run([sys.executable, 'verificar.py'], capture_output=True, text=True, encoding='utf-8')
print(resultado.stdout, resultado.stderr)